In [ ]:
# ==============================================================================
# 🚀 SCRIPT DE FINE-TUNING : MISTRAL 7B (Version CNRS)
# ==============================================================================

# 1. INSTALLATION DE UNSLOTH (L'accélérateur d'entraînement)
# ----------------------------------------------------------
print("⏳ Installation des librairies... (2-3 minutes)")
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# 2. CHARGEMENT DU MODÈLE MISTRAL
# -------------------------------
# On utilise la version v0.3 optimisée (4-bit) pour qu'elle rentre dans le Colab gratuit
model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
max_seq_length = 2048

print(f"🚀 Chargement de {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Configuration LoRA (C'est la partie "mémoire" qu'on va entraîner)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 3. CHARGEMENT DE TON DATASET
# ----------------------------
fichier_donnees = "dataset_cnrs_FINAL_HYBRIDE.jsonl"

# Le "Template" : On explique à Mistral comment lire tes données
alpaca_prompt = """Ci-dessous une instruction qui décrit une tâche, accompagnée d'un contexte. Écrivez une réponse qui complète correctement la demande.

### Instruction:
{}

### Entrée:
{}

### Réponse:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # On remplit le template
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

print("📂 Préparation des données...")
try:
    dataset = load_dataset("json", data_files = fichier_donnees, split = "train")
    dataset = dataset.map(formatting_prompts_func, batched = True)
    print(f"✅ {len(dataset)} exemples chargés avec succès !")
except Exception as e:
    print(f"❌ ERREUR : Impossible de lire le fichier {fichier_donnees}. Vérifie qu'il est bien uploadé à gauche !")
    raise e

# 4. LANCEMENT DE L'ENTRAÎNEMENT
# ------------------------------
print("🏋️‍♂️ DÉBUT DU FINE-TUNING (Cela va prendre 10 à 15 minutes)...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 60 étapes suffisent pour ~90 questions
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

print("🎉 ENTRAÎNEMENT TERMINÉ ! Mistral connait maintenant les règles du CNRS.")

# 5. SAUVEGARDE DU MODÈLE ENTRAÎNÉ
# --------------------------------
nom_dossier_modele = "mistral_cnrs_finetuned"
model.save_pretrained(nom_dossier_modele)
tokenizer.save_pretrained(nom_dossier_modele)
print(f"💾 Modèle sauvegardé dans le dossier '{nom_dossier_modele}'.")

⏳ Installation des librairies... (2-3 minutes)
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-mva37_5g/unsloth_dfa31415cfe24f2cb6ec2d1edd9c0ddb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-mva37_5g/unsloth_dfa31415cfe24f2cb6ec2d1edd9c0ddb
  Resolved https://github.com/unslothai/unsloth.git to commit 29e3b7c560f205572b60bd4e9fa976882136b583
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.747400
2,1.631700
3,1.630100
4,1.603000
5,1.485900
6,1.310100
7,1.184500
8,1.116100
9,1.087100
10,0.997900


wandb: WARNING URL not available in offline run


train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,▄▃▂▂▂▃▂▃▂▃▂▁▂▂▄▃▅▃▃▅▃▃▅▅▃▂▄▂▃▂▃▃▂▂▂▁▁▂▃█
train/learning_rate,▁▂▅███▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▁
train/loss,███▇▆▆▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▂
total_flos,1.5939516463546368e+16
train/epoch,5
train/global_step,60
train/grad_norm,2.89855
train/learning_rate,0.0
train/loss,0.2481


🎉 ENTRAÎNEMENT TERMINÉ ! Mistral connait maintenant les règles du CNRS.
💾 Modèle sauvegardé dans le dossier 'mistral_cnrs_finetuned'.


In [ ]:
# ==========================================================
# SAUVEGARDE DÉFINITIVE SUR GOOGLE DRIVE
# ==========================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# Nom du dossier dans ton Drive
dossier_drive = "/content/drive/MyDrive/Modele_Mistral_CNRS_Final"

print(f"💾 Fusion du modèle et sauvegarde vers : {dossier_drive}...")
print("(Cela peut prendre 5 à 10 minutes, ne ferme pas !)")

# On sauvegarde en format "merged_16bit"
# C'est LE format universel qui marchera dans ton autre notebook
model.save_pretrained_merged(
    dossier_drive,
    tokenizer,
    save_method = "merged_16bit",
)

print("✅ SUCCÈS ! Le modèle est sécurisé sur ton Drive.")

Mounted at /content/drive
💾 Fusion du modèle et sauvegarde vers : /content/drive/MyDrive/Modele_Mistral_CNRS_Final...
(Cela peut prendre 5 à 10 minutes, ne ferme pas !)


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00003.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files:  67%|██████▋   | 2/3 [04:00<01:57, 117.62s/it]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [11:14<00:00, 224.85s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/Modele_Mistral_CNRS_Final`
✅ SUCCÈS ! Le modèle est sécurisé sur ton Drive.
